# Gemini Embedding 2 & Vector Search 2.0 Workshop 🔑

---

# [Part 1] 멀티모달 임베딩부터 Vector Search 2.0 검색까지 (약 30분)

먼저 검색 엔진 내부의 핵심 연산 과정을 NumPy로 직접 구현해 보고, **동일한 질의를 Vector Search 2.0에 요청하여 결과를 나란히 비교**합니다.

* **1단계** 비디오 청킹 (FFmpeg Stream Copy)
* **2단계** Dense 임베딩 · 코사인 유사도 · t-SNE 궤적 · 크로스모달 검색
* **3단계** 로컬 하이브리드 (SimpleBM25 + `alpha`)
* **4단계** Vector Search 2.0 — kNN · RRF 하이브리드 · 메타데이터 필터
* **5단계** Ranking API 리랭킹 · 비디오 크라우딩 필터
* **6단계** 자원 정리

> [!WARNING]
> **설치 셀을 실행하면 주피터 커널이 1회 자동 재시작됩니다.**
> NumPy 등 바이너리 패키지를 새 버전으로 다시 로드하기 위한 정상 동작입니다.
> 커널 상태가 `idle` 이 되면 **설치 셀부터** 다시 실행해 주세요. 두 번째 실행부터는 재시작하지 않습니다.

In [ ]:
# Install necessary libraries (최초 1회, 약 40초)
%pip install -qU google-genai google-cloud-vectorsearch google-cloud-discoveryengine google-cloud-storage Pillow opencv-python imageio-ffmpeg numpy scikit-learn matplotlib seaborn

# NumPy/SciPy 같은 바이너리 패키지는 업그레이드해도 "이미 메모리에 올라간 구버전"이 계속 쓰입니다.
# import가 성공하더라도 뒤쪽 셀에서 ABI 불일치로 깨지므로, 설치 직후 커널을 무조건 1회 재시작합니다.
import os
import IPython

_RESTART_FLAG = "/tmp/.workshop_kernel_restarted"

if os.path.exists(_RESTART_FLAG):
    print("✅ 커널 재시작이 이미 완료된 세션입니다. 다음 셀로 진행하세요.")
else:
    with open(_RESTART_FLAG, "w"):
        pass
    print("🚨 주피터 커널을 자동으로 재시작합니다.")
    print("   재시작이 끝나면 (좌측 상단 커널 상태가 idle이 되면) 이 셀부터 다시 실행해 주세요.")
    IPython.Application.instance().kernel.do_shutdown(True)

### 인증 및 클라이언트 초기화 (Authentication)

임베딩 및 검색 호출 모두 ADC(Application Default Credentials)를 사용하므로 별도의 API Key가 필요하지 않습니다.
Gemini Embedding 2는 Vertex AI **`global` 엔드포인트**에서 제공되므로 임베딩 클라이언트는 `location="global"`로 초기화합니다 (Vector Search 컬렉션은 `us-central1`).

In [ ]:
import os
import subprocess
import google.auth
from google import genai


# 1. Project ID 자동 감지
def get_project_id():
    """활성화된 Google Cloud Project ID를 자동으로 감지합니다."""
    try:
        _, project_id = google.auth.default()
        if project_id:
            return project_id
    except Exception:
        pass
    return subprocess.check_output(["gcloud", "config", "get-value", "project"]).decode("utf-8").strip()


PROJECT_ID = get_project_id()
REGION = "us-central1"                 # Vector Search 2.0 컬렉션 리전
EMBEDDING_LOCATION = "global"          # Gemini Embedding 2 제공 엔드포인트
SOURCE_BUCKET = "ai-multimodal-data"   # 실습용 공용 읽기 전용 버킷
MODEL_ID = "gemini-embedding-2"

# 2. GenAI 클라이언트 — Vertex AI(ADC) 경로. API Key 불필요.
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=EMBEDDING_LOCATION,
)

print(f"Google Cloud Project ID: {PROJECT_ID}")
print(f"컬렉션 리전: {REGION} / 임베딩 엔드포인트: {EMBEDDING_LOCATION} / 모델: {MODEL_ID}")


## 0단계: Vector Search 2.0 컬렉션 만들기

서버리스 컬렉션을 생성합니다 (약 10초 소요).
`description`은 `text_search` 대상 데이터 필드, `tag`는 `{"tag": {"$eq": "Me"}}` 필터 대상 데이터 필드로 정의합니다.

In [ ]:
import time
from google.cloud import vectorsearch_v1beta as vectorsearch
from google.api_core import exceptions

COLLECTION_ID = "multimodal-media-collection"
COLLECTION_PARENT = f"projects/{PROJECT_ID}/locations/{REGION}"
COLLECTION_NAME = f"{COLLECTION_PARENT}/collections/{COLLECTION_ID}"

vector_search_client = vectorsearch.VectorSearchServiceClient()

# 데이터 필드: description -> text_search 대상, tag -> filter 대상, source -> 크라우딩 필터 기준
data_schema = {
    "type": "object",
    "properties": {
        "description": {"type": "string"},
        "tag": {"type": "string"},
        "media_type": {"type": "number"},
        "source": {"type": "string"},
    },
}

# 벡터 필드: 3072차원 dense 하나.
# vertex_embedding_config를 붙여두면 검색 시 search_text만 넘겨도 서버가 질의 임베딩을 대신 생성합니다.
vector_schema = {
    "content_embedding": vectorsearch.VectorField(
        dense_vector=vectorsearch.DenseVectorField(
            dimensions=3072,
            vertex_embedding_config=vectorsearch.VertexEmbeddingConfig(
                model_id="gemini-embedding-2"
            ),
        )
    )
}

collection_config = vectorsearch.Collection(
    data_schema=data_schema,
    vector_schema=vector_schema,
)

started_at = time.time()
try:
    operation = vector_search_client.create_collection(
        request=vectorsearch.CreateCollectionRequest(
            parent=COLLECTION_PARENT,
            collection_id=COLLECTION_ID,
            collection=collection_config,
        )
    )
    collection = operation.result()
    print(f"✅ 컬렉션 생성 완료 ({time.time() - started_at:.0f}초): {collection.name}")
except exceptions.AlreadyExists:
    collection = vector_search_client.get_collection(name=COLLECTION_NAME)
    print(f"ℹ️ 동일한 ID의 컬렉션이 이미 존재하여 그대로 재사용합니다: {COLLECTION_ID}")

## 1단계: 비디오 전처리 및 세그먼트 분할 (Chunking)

긴 영상 전체를 하나의 임베딩으로 생성하면 세부 정보가 희석될 수 있습니다. FFmpeg 스트림 복사(`-c copy`)를 통해 재인코딩 없이 10초 단위 세그먼트로 분할합니다.
(소스: `gs://ai-multimodal-data/team_usa_tech.mp4`)

In [ ]:
import os
import cv2
import subprocess
import imageio_ffmpeg
from google.cloud import storage

VIDEO_BLOB = "team_usa_tech.mp4"
CHUNK_DURATION = 10   # seconds
CHUNK_DIR = "local_chunks"

# 1. 원본 영상 다운로드 (이미 있으면 건너뜀)
local_video_path = VIDEO_BLOB
if not os.path.exists(local_video_path):
    print(f"비디오 다운로드 중: gs://{SOURCE_BUCKET}/{VIDEO_BLOB} ...")
    storage.Client().bucket(SOURCE_BUCKET).blob(VIDEO_BLOB).download_to_filename(local_video_path)
print(f"원본 영상 준비 완료: {local_video_path}")

# 2. FFmpeg 스트림 복사 분할
os.makedirs(CHUNK_DIR, exist_ok=True)
for f in os.listdir(CHUNK_DIR):
    if f.endswith(".mp4"):
        os.remove(os.path.join(CHUNK_DIR, f))

ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()
print("FFmpeg 스트림 복사(재인코딩 없음) 방식으로 비디오 분할(Chunking)을 시작합니다...")
subprocess.run(
    [
        ffmpeg_bin, "-y", "-i", local_video_path,
        "-c", "copy",
        "-f", "segment",
        "-segment_time", str(CHUNK_DURATION),
        "-reset_timestamps", "1",
        os.path.join(CHUNK_DIR, "chunk_%d.mp4"),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True,
)

# 3. 청크별 정밀 메타데이터(시작/종료 시각) 산출
chunk_metadata = []
chunk_files = sorted(
    [f for f in os.listdir(CHUNK_DIR) if f.startswith("chunk_") and f.endswith(".mp4")],
    key=lambda x: int(x.split("_")[1].split(".")[0]),
)
for i, chunk_name in enumerate(chunk_files):
    chunk_file = os.path.join(CHUNK_DIR, chunk_name)
    cap = cv2.VideoCapture(chunk_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = total_frames / fps if fps > 0 and total_frames > 0 else CHUNK_DURATION
    chunk_metadata.append({
        "chunk_id": i,
        "file_path": chunk_file,
        "start_time": i * CHUNK_DURATION,
        "end_time": i * CHUNK_DURATION + duration,
    })

print(f"정밀 메타데이터를 포함한 {len(chunk_metadata)}개의 비디오 청크를 생성했습니다.")
print("메타데이터 샘플:", chunk_metadata[0])

## 2단계: 멀티모달 임베딩 생성 (Generating Embeddings)

`gemini-embedding-2`는 텍스트, 이미지, 비디오를 **동일한 3072차원 벡터 공간**에 매핑합니다.
범용 임베딩 함수를 정의하고 10개의 비디오 청크를 `ThreadPoolExecutor`로 병렬 처리합니다.

In [ ]:
from google.genai import types


def generate_multimodal_embedding(content_path, content_type):
    """Gemini Embedding 2로 텍스트/이미지/비디오를 3072차원 벡터로 변환합니다."""
    if content_type == "text":
        contents = content_path
    else:
        mime_type = "video/mp4" if content_type == "video" else "image/jpeg"
        with open(content_path, "rb") as f:
            contents = types.Part.from_bytes(data=f.read(), mime_type=mime_type)

    response = client.models.embed_content(
        model=MODEL_ID,
        contents=contents,
        config=types.EmbedContentConfig(
            output_dimensionality=3072,
            http_options=types.HttpOptions(
                retry_options=types.HttpRetryOptions(
                    attempts=10,
                    initial_delay=1.0,
                    max_delay=3.0,
                )
            ),
        ),
    )
    return response.embeddings[0].values


print("멀티모달 임베딩 생성 함수 정의 완료")

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

MAX_WORKERS = 8   # 동시 요청 수. 너무 크게 올리면 스로틀링이 걸려 오히려 느려집니다
LIMIT = 10        # 로컬 실습용으로 앞쪽 10개 청크만 사용

valid_chunks = chunk_metadata[:LIMIT]
print(f"상위 {len(valid_chunks)}개 비디오 청크의 밀집 임베딩 벡터를 병렬 생성합니다 (max_workers={MAX_WORKERS})...")

start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    chunk_embeddings = list(
        executor.map(lambda item: generate_multimodal_embedding(item["file_path"], "video"), valid_chunks)
    )
for item, embedding in zip(valid_chunks, chunk_embeddings):
    item["dense_embedding"] = embedding

print(f"{len(valid_chunks)}개 청크 임베딩 생성 완료 ({time.time() - start:.1f}초 소요)")
print(f"벡터 차원: {len(valid_chunks[0]['dense_embedding'])}")

### 📊 glass box #1: 코사인 유사도와 t-SNE 시간축 궤적

NumPy로 코사인 유사도를 직접 계산하여 기준 청크와 가장 유사한 장면 및 가장 거리가 먼 장면을 탐색합니다.
세 청크를 **인라인 플레이어로 나란히 재생**하여 유사도 수치와 실제 비디오 장면을 대조 확인합니다. 이어서 3072차원 임베딩을 t-SNE로 2차원 축소하여 시간 순 궤적을 시각화합니다.

In [ ]:
# ==========================================
# 1. Embedding Distance & Similarity
# ==========================================
import base64
import os
import subprocess
import tempfile

import imageio_ffmpeg
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML

PREVIEW_WIDTH = 480   # 미리보기 재인코딩 가로 해상도(px)
PREVIEW_CRF = 30      # 값이 클수록 더 압축됩니다 (용량 ↓ / 화질 ↓)


def cosine_similarity(v1, v2):
    """두 벡터의 코사인 유사도를 numpy.dot으로 직접 계산합니다."""
    a = np.array(v1)
    b = np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def to_preview_video(video_path):
    """청크(AV1 1080p, 약 2MB)를 480px H.264(약 300KB)로 재인코딩해 바이트로 돌려줍니다.

    노트북에 base64로 직접 심으려면 용량을 줄여야 하고, H.264가 브라우저 호환 범위가 가장 넓습니다.
    OpenCV 내장 FFmpeg에는 AV1 디코더가 없으므로 여기서도 imageio-ffmpeg 바이너리를 사용합니다.
    """
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        preview_path = tmp.name
    subprocess.run(
        [
            imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-v", "error", "-i", video_path,
            "-vf", f"scale={PREVIEW_WIDTH}:-2",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", str(PREVIEW_CRF),
            "-an", "-movflags", "+faststart", preview_path,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )
    with open(preview_path, "rb") as f:
        data = f.read()
    os.remove(preview_path)
    return data


def show_chunk_videos(chunks, captions, width=300):
    """여러 청크를 가로로 나란히 인라인 플레이어로 띄웁니다.

    정지 프레임만 보면 "얼마나 비슷한지"를 판단하기 어려우므로 직접 재생해서 비교합니다.
    로컬 파일은 상대 경로로 참조하면 프론트엔드에 따라 404가 나므로 base64 data URI로 심습니다.
    """
    start = time.time()
    with ThreadPoolExecutor(max_workers=len(chunks)) as executor:
        clips = list(executor.map(to_preview_video, [chunk["file_path"] for chunk in chunks]))

    cells = ""
    for clip, caption in zip(clips, captions):
        encoded = base64.b64encode(clip).decode("ascii")
        cells += (
            "<td style='padding:6px;text-align:center;vertical-align:top'>"
            f"<video width='{width}' controls preload='metadata' style='border-radius:6px;background:#000'>"
            f"<source src='data:video/mp4;base64,{encoded}' type='video/mp4'></video>"
            f"<div style='font-size:90%;padding-top:6px'>{caption}</div></td>"
        )
    display(HTML(f"<table style='border-collapse:collapse'><tr>{cells}</tr></table>"))
    print(f"(미리보기 {len(chunks)}건 480px 재인코딩: {time.time() - start:.1f}초)")


print("=== 로컬 임베딩 코사인 유사도 연산 분석 (DB 연결 없이 진행) ===")
ref_chunk = valid_chunks[0]
similarities = sorted(
    [(item, cosine_similarity(ref_chunk["dense_embedding"], item["dense_embedding"])) for item in valid_chunks[1:]],
    key=lambda x: x[1],
    reverse=True,
)

most_similar, max_sim = similarities[0]
least_similar, min_sim = similarities[-1]

print(f"기준 세그먼트 (Reference): Chunk {ref_chunk['chunk_id']} ({ref_chunk['start_time']}초 - {ref_chunk['end_time']:.1f}초)\n")
print(f"✅ [가장 유사한 세그먼트] Chunk {most_similar['chunk_id']} ({most_similar['start_time']}초 - {most_similar['end_time']:.1f}초) | 코사인 유사도 {max_sim:.4f}")
print(f"❌ [가장 이질적인 세그먼트] Chunk {least_similar['chunk_id']} ({least_similar['start_time']}초 - {least_similar['end_time']:.1f}초) | 코사인 유사도 {min_sim:.4f}")

show_chunk_videos(
    [ref_chunk, most_similar, least_similar],
    [
        f"기준 · Chunk {ref_chunk['chunk_id']} ({ref_chunk['start_time']}초~)",
        f"✅ 가장 유사 · Chunk {most_similar['chunk_id']} (코사인 {max_sim:.3f})",
        f"❌ 가장 이질적 · Chunk {least_similar['chunk_id']} (코사인 {min_sim:.3f})",
    ],
)

In [ ]:
# ===================================================
# 2. t-SNE Dimensionality Reduction & Trajectory Plot
# ===================================================
import seaborn as sns
from sklearn.manifold import TSNE

print("비디오 청크 임베딩에 대한 t-SNE 차원 축소(3072 -> 2)를 수행 중입니다...")

embeddings_matrix = np.array([item["dense_embedding"] for item in valid_chunks])
chronological_labels = [f"{item['start_time']}s - {item['end_time']:.0f}s" for item in valid_chunks]
chunk_indices = list(range(len(valid_chunks)))

tsne = TSNE(n_components=2, perplexity=min(5, len(valid_chunks) - 1), random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_matrix)

plt.figure(figsize=(10, 8))
sns.set_theme(style="whitegrid")

scatter = plt.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    c=chunk_indices,
    cmap="viridis",
    s=150,
    zorder=3,
    edgecolors="black",
    linewidth=1.5,
)

# 시간 순서대로 좌표를 연결하여 '비디오 궤적'을 그립니다.
for i in range(len(embeddings_2d) - 1):
    plt.annotate(
        "",
        xy=(embeddings_2d[i + 1, 0], embeddings_2d[i + 1, 1]),
        xytext=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
        arrowprops=dict(arrowstyle="->", color="red", lw=1.5, ls="--", alpha=0.6, connectionstyle="arc3,rad=0.1"),
    )

for idx, (x, y) in enumerate(embeddings_2d):
    plt.text(
        x + 0.2,
        y + 0.2,
        f"Chunk {idx}\n({chronological_labels[idx]})",
        fontsize=9,
        weight="bold",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", edgecolor="gray", alpha=0.8),
        zorder=4,
    )

plt.colorbar(scatter, label="Chronological Chunk Index")
plt.title("Chronological Video Embedding Trajectory (t-SNE Projections)", fontsize=14, weight="bold")
plt.xlabel("t-SNE Dimension 1", fontsize=11)
plt.ylabel("t-SNE Dimension 2", fontsize=11)
plt.tight_layout()
plt.show()

print("\n💡 해석 가이드:")
print("- 뭉쳐 있는 구간: 장면 흐름이 정적이거나 연속적입니다.")
print("- 크게 도약하는 선: 씬 전환, 페이드, 카메라가 크게 움직인 지점입니다.")
print("- 이전 좌표로 되돌아오는 궤적: 동일 씬의 반복(예: 스튜디오 화면 복귀)을 암시합니다.")

### 🔍 Dense 단독 시맨틱 검색 — 크로스모달

별도의 텍스트 메타데이터(자막, 키워드, 파일명 등) 없이, **텍스트 질의 벡터와 비디오 청크 벡터 간의 거리(코사인 유사도)만으로** 관련 영상 구간을 검색합니다.

In [ ]:
DENSE_QUERY = "athletes training in the snow"   # 원하는 질의로 바꿔 보세요 (한국어 질의도 동작합니다)

query_dense = generate_multimodal_embedding(DENSE_QUERY, "text")
dense_scores = [cosine_similarity(query_dense, item["dense_embedding"]) for item in valid_chunks]
ranked_indices = np.argsort(dense_scores)[::-1][:3]

print(f"🔍 [Dense 단독 시맨틱 검색] 질의: '{DENSE_QUERY}'")
print("   설명문 없이 '텍스트 벡터 ↔ 영상 벡터' 거리만으로 매칭합니다.\n")
for rank, idx in enumerate(ranked_indices, start=1):
    item = valid_chunks[idx]
    print(f"[{rank}위] 코사인 유사도 {dense_scores[idx]:.4f} | Chunk {item['chunk_id']} ({item['start_time']}초 - {item['end_time']:.1f}초)")

show_chunk_videos(
    [valid_chunks[idx] for idx in ranked_indices],
    [f"{rank}위 · Chunk {valid_chunks[idx]['chunk_id']} (코사인 {dense_scores[idx]:.3f})" for rank, idx in enumerate(ranked_indices, start=1)],
)

## 3단계: 로컬 하이브리드 검색 구현 (glass box #2)

밀집(Dense) 임베딩 기반 검색은 고유명사나 특정 키워드의 정확한 일치에 취약할 수 있으므로, 실제 검색 시스템에서는 희소(Sparse/BM25) 점수를 결합한 하이브리드 방식을 널리 활용합니다.

1. **Gemini Flash 캡션** — 청크별 영어 설명문 병렬 생성 (희소 검색의 입력 텍스트)
2. **SimpleBM25** — 순수 파이썬 BM25 직접 구현
3. **`alpha` 결합** — `alpha * dense + (1 - alpha) * sparse`

In [ ]:
import logging

# google-genai가 generate_content 호출 때마다 남기는 AFC 권고 로그를 끕니다 (이 실습은 함수 호출을 쓰지 않습니다).
logging.getLogger("google_genai").setLevel(logging.ERROR)


def generate_chunk_description(video_path):
    """Gemini Flash로 10초 클립의 짧은 설명문을 생성합니다 (희소/전문 검색용 텍스트)."""
    with open(video_path, "rb") as f:
        part = types.Part.from_bytes(data=f.read(), mime_type="video/mp4")
    response = client.models.generate_content(
        model="gemini-3.7-flash",
        contents=[
            part,
            "Provide a short, detailed description of what is happening in this 10-second video clip. Focus on keywords, actions, and objects.",
        ],
        config=types.GenerateContentConfig(
            http_options=types.HttpOptions(
                retry_options=types.HttpRetryOptions(
                    attempts=10,
                    initial_delay=1.0,
                    max_delay=3.0,
                )
            )
        ),
    )
    return response.text


print(f"Gemini Flash로 {len(valid_chunks)}개 청크의 설명문을 병렬 생성합니다 (max_workers={MAX_WORKERS})...")
start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    chunk_descriptions = list(
        executor.map(lambda item: generate_chunk_description(item["file_path"]), valid_chunks)
    )
for item, description in zip(valid_chunks, chunk_descriptions):
    item["description"] = description.strip()

print(f"설명문 생성 완료 ({time.time() - start:.1f}초 소요)\n")
print("샘플 (Chunk 0):", valid_chunks[0]["description"][:200], "...")

In [ ]:
import math
from collections import Counter


class SimpleBM25:
    """교육용 BM25 구현. 검색 엔진의 희소(Sparse) 점수가 어떻게 계산되는지 보여줍니다."""

    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.avgdl = sum(len(doc) for doc in corpus) / self.corpus_size if self.corpus_size > 0 else 0
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []

        for doc in corpus:
            self.doc_len.append(len(doc))
            self.doc_freqs.append(Counter(doc))

        for doc in corpus:
            for word in set(doc):
                self.idf[word] = self.idf.get(word, 0) + 1

        for word, freq in self.idf.items():
            self.idf[word] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1)

    def get_scores(self, query):
        scores = []
        for i in range(self.corpus_size):
            score = 0
            doc_freq = self.doc_freqs[i]
            doc_len = self.doc_len[i]
            for word in query:
                if word in doc_freq:
                    idf = self.idf.get(word, 0)
                    freq = doc_freq[word]
                    numerator = freq * (self.k1 + 1)
                    denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                    score += idf * numerator / denominator
            scores.append(score)
        return scores


def tokenize(text):
    return text.lower().split()


bm25 = SimpleBM25([tokenize(item["description"]) for item in valid_chunks])
print(f"{len(valid_chunks)}개 문서 기반 희소 검색용 BM25 인덱스를 구축했습니다.")

In [ ]:
def hybrid_search(query_text, alpha=0.7, top_k=3):
    """Dense 코사인 점수와 BM25 희소 점수를 alpha 가중치로 선형 결합합니다.

    alpha=1.0 이면 순수 시맨틱, alpha=0.0 이면 순수 키워드 검색이 됩니다.
    """

    def normalize(scores):
        # 두 점수는 스케일이 완전히 다르므로 [0, 1]로 정규화한 뒤에 더해야 합니다.
        if scores.max() == scores.min():
            return np.zeros_like(scores)
        return (scores - scores.min()) / (scores.max() - scores.min())

    query_embedding = generate_multimodal_embedding(query_text, "text")
    dense_scores = normalize(np.array([cosine_similarity(query_embedding, item["dense_embedding"]) for item in valid_chunks]))
    sparse_scores = normalize(np.array(bm25.get_scores(tokenize(query_text))))
    combined_scores = alpha * dense_scores + (1 - alpha) * sparse_scores

    return [
        {
            "chunk": valid_chunks[idx],
            "score": combined_scores[idx],
            "dense_score": dense_scores[idx],
            "sparse_score": sparse_scores[idx],
        }
        for idx in np.argsort(combined_scores)[::-1][:top_k]
    ]


HYBRID_QUERY = "olympic athlete interview"

for alpha in (0.8, 0.3):
    label = "밀집 시맨틱 중심" if alpha > 0.5 else "희소 키워드 중심"
    print(f"\n=== alpha = {alpha} ({label}) | 질의: '{HYBRID_QUERY}' ===")
    for rank, res in enumerate(hybrid_search(HYBRID_QUERY, alpha=alpha), start=1):
        chunk = res["chunk"]
        print(f"[{rank}위] 최종 {res['score']:.3f} (dense {res['dense_score']:.3f} / sparse {res['sparse_score']:.3f}) | Chunk {chunk['chunk_id']}")
        print(f"       {chunk['description'][:110]}...")

## 4단계: 스케일 업 — 사전 연산 레지스트리 불러오기 · 개인화 태깅

사전 임베딩이 완료된 데이터 레지스트리(이미지 4,606건 + 비디오 청크 199건)를 다운로드하여 **약 1,000건으로 서브샘플링**합니다.
개인화 검색 실습을 위해 이미지 앞 30장에 `tag="Me"`를 **독립된 데이터 필드**로 부여합니다.

In [ ]:
import pickle
import urllib.request
from html import escape
from urllib.parse import quote
from IPython.display import display, HTML

REGISTRY_URL = "https://storage.googleapis.com/ai-multimodal-data/full_dataset_registry.pkl"
REGISTRY_FILE = "full_dataset_registry.pkl"
IMAGE_LIMIT = 800   # 업서트할 이미지 수 (비디오 청크는 전량 사용)

TAG_ME = "Me"
TAG_PUBLIC = "Public"
ME_TAG_COUNT = 30   # 'Me' 태그를 붙일 이미지 수 (필터 결과가 '나열'이 아니라 '재정렬'로 보이도록)
TARGET_IDS = [      # 태깅 결과를 눈으로 확인할 미리보기 3장
    "1003163366_44323f5815.jpg",
    "1007129816_e794419615.jpg",
    "1015118661_980735411b.jpg",
]

if not os.path.exists(REGISTRY_FILE):
    print(f"사전 연산된 레지스트리를 다운로드합니다 (135MB): {REGISTRY_URL}")
    urllib.request.urlretrieve(REGISTRY_URL, REGISTRY_FILE)

with open(REGISTRY_FILE, "rb") as f:
    full_registry = pickle.load(f)
print(f"전체 레지스트리 항목 수: {len(full_registry)}")

# 1. 서브샘플링 (미리보기 3장은 이미지 맨 앞에 오도록 먼저 뽑습니다)
video_items = [item for item in full_registry if item["type"] == "video_chunk"]
tagged_items = [item for item in full_registry if item["id"] in TARGET_IDS]
other_images = [item for item in full_registry if item["type"] == "image" and item["id"] not in TARGET_IDS]
combined_registry = video_items + tagged_items + other_images[: IMAGE_LIMIT - len(tagged_items)]

# 2. 데이터 필드 부여: tag(필터용) / source(크라우딩용) / dp_id(VS2 오브젝트 ID)
#    'Me'는 이미지 앞 ME_TAG_COUNT장에 붙입니다. 미리보기 3장을 앞에 배치했으므로 항상 포함됩니다.
image_items = [item for item in combined_registry if item["type"] == "image"]
me_ids = {item["id"] for item in image_items[:ME_TAG_COUNT]}

for item in combined_registry:
    item["tag"] = TAG_ME if item["id"] in me_ids else TAG_PUBLIC
    if item["type"] == "video_chunk":
        item["source"] = item.get("source_video", item["id"].rsplit("_", 1)[0])
    else:
        item["source"] = "flickr8k"
    item["dp_id"] = item["id"].replace(".", "_").replace("-", "_").replace(" ", "_")

registry_by_dp_id = {item["dp_id"]: item for item in combined_registry}
print(f"업서트 대상: 총 {len(combined_registry)}건 (video_chunk {len(video_items)} / image {len(image_items)})")

# 3. 로컬 인메모리 인덱스도 같은 서브샘플로 재구성 (뒤의 '3가지 검색 방식 비교' 셀에서 사용)
registry_matrix = np.array([item["dense_embedding"] for item in combined_registry], dtype=np.float32)
registry_matrix /= np.linalg.norm(registry_matrix, axis=1, keepdims=True)
combined_bm25 = SimpleBM25([tokenize(item["description"]) for item in combined_registry])
print(f"인메모리 벡터 행렬 {registry_matrix.shape} + 통합 BM25 인덱스 구축 완료")


# 4. 결과 렌더러
def gcs_public_url(gcs_path):
    """gs:// 경로를 HTTPS URL로 변환합니다. 실습 버킷은 공개 읽기 전용이라 인증 없이 바로 표시됩니다."""
    bucket_name, blob_name = gcs_path[5:].split("/", 1)
    return f"https://storage.googleapis.com/{bucket_name}/{quote(blob_name)}"


def media_html(dp_id, width=160):
    """검색 결과 한 건을 이미지 썸네일 또는 인라인 비디오 플레이어로 렌더링합니다.

    다운로드 없이 공개 URL을 그대로 씁니다. 비디오 URL 뒤의 '#t=1'은 브라우저가 1초 지점 프레임을
    포스터로 잡아 주므로, 재생 전에도 어떤 장면인지 알 수 있습니다.
    """
    item = registry_by_dp_id.get(dp_id)
    if item is None:
        return ""
    url = gcs_public_url(item["path"])
    if item["type"] == "image":
        return f"<img src='{url}' width='{width}' style='border-radius:4px'>"
    return (
        f"<video src='{url}#t=1' width='{width}' controls preload='metadata'"
        " style='border-radius:4px;background:#000'></video>"
    )


def field(data_object, name, default=""):
    return data_object.data[name] if name in data_object.data else default


def vs2_rows(results):
    """Vector Search 2.0 검색 응답을 렌더링용 dict 리스트로 변환합니다."""
    return [
        {
            "id": result.data_object.data_object_id,
            "score": result.distance,
            "description": field(result.data_object, "description"),
            "tag": field(result.data_object, "tag"),
            "media_type": field(result.data_object, "media_type", 1),
            "source": field(result.data_object, "source"),
        }
        for result in results
    ]


def local_rows(items, scores):
    """로컬 검색 결과를 vs2_rows와 동일한 형태로 맞춰 나란히 비교할 수 있게 합니다."""
    return [
        {
            "id": item["dp_id"],
            "score": float(score),
            "description": item["description"],
            "tag": item["tag"],
            "media_type": 1 if item["type"] == "image" else 0,
            "source": item["source"],
        }
        for item, score in zip(items, scores)
    ]


def show_results(rows, title, preview=3):
    """검색 결과를 순위표로 출력합니다. 상위 preview건에는 썸네일 또는 인라인 플레이어를 붙입니다."""
    html = [
        f"<div style='font-weight:600;margin:12px 0 4px'>[{escape(title)}]</div>",
        "<table style='border-collapse:collapse;text-align:left'>",
    ]
    for rank, row in enumerate(rows, start=1):
        kind = "image" if row["media_type"] == 1 else "video_chunk"
        cell = "border-top:1px solid #eee;padding:6px 10px;vertical-align:top"
        html.append(
            f"<tr><td style='{cell}'>{rank}위</td>"
            f"<td style='{cell}'>{media_html(row['id']) if rank <= preview else ''}</td>"
            f"<td style='{cell};max-width:560px'>"
            f"<code>{escape(row['id'])}</code>"
            f" <span style='color:#999'>{kind} · tag={escape(str(row['tag']))} · score {row['score']:.4f}</span>"
            f"<div style='font-size:90%;color:#444'>{escape(row['description'][:160])}…</div>"
            "</td></tr>"
        )
    display(HTML("".join(html) + "</table>"))


print(f"\n🏷️ '{TAG_ME}' 태그 부여 완료: 이미지 {len(me_ids)}장 (아래는 그중 미리보기 {len(tagged_items)}장)")
display(HTML(
    "<table style='border-collapse:collapse'><tr>"
    + "".join(
        "<td style='padding:6px;text-align:center;vertical-align:top'>"
        f"<img src='{gcs_public_url(item['path'])}' width='180' style='border-radius:4px'>"
        f"<div style='font-size:85%;max-width:180px'>{escape(item['description'][:70])}…</div></td>"
        for item in tagged_items
    )
    + "</tr></table>"
))

### 병렬 배치 업서트

`BatchCreateDataObjects` 1회 요청 상한(250건)에 맞추어 여러 배치로 분할한 후 병렬로 업서트합니다.

> Part 1에서는 **ANN 인덱스를 생성하지 않습니다.** 인덱스가 없더라도 Vector Search 2.0은 kNN 완전탐색 방식으로 검색·필터·전문검색을 정상 수행합니다.

In [ ]:
UPSERT_WORKERS = 8   # 배치 전송 동시성
BATCH_SIZE = 250     # BatchCreateDataObjects 1회 요청 상한

data_client = vectorsearch.DataObjectServiceClient()
search_client = vectorsearch.DataObjectSearchServiceClient()

# 1. 업서트 요청 조립
upsert_requests = [
    vectorsearch.CreateDataObjectRequest(
        parent=collection.name,
        data_object_id=item["dp_id"],
        data_object=vectorsearch.DataObject(
            data={
                "description": item["description"],
                "tag": item["tag"],
                "media_type": 1 if item["type"] == "image" else 0,
                "source": item["source"],
            },
            vectors={
                "content_embedding": vectorsearch.Vector(
                    dense=vectorsearch.DenseVector(values=item["dense_embedding"])
                )
            },
        ),
    )
    for item in combined_registry
]
batches = [upsert_requests[i : i + BATCH_SIZE] for i in range(0, len(upsert_requests), BATCH_SIZE)]


def send_batch(batch):
    try:
        data_client.batch_create_data_objects(
            request=vectorsearch.BatchCreateDataObjectsRequest(parent=collection.name, requests=batch)
        )
    except exceptions.AlreadyExists:
        pass   # 노트북 재실행 시 이미 올라간 오브젝트는 건너뜁니다
    return len(batch)


print(f"{len(batches)}개 배치를 병렬 업서트합니다 (max_workers={UPSERT_WORKERS})...")
start = time.time()
with ThreadPoolExecutor(max_workers=UPSERT_WORKERS) as executor:
    upserted = sum(executor.map(send_batch, batches))
print(f"{upserted}개 데이터 오브젝트 업서트 완료 ({time.time() - start:.1f}초 소요)")


### ⭐ [핵심] 3가지 검색 방식의 비교

| | 검색 주체 | 질의 임베딩 | 유사도 연산 |
| :--- | :--- | :--- | :--- |
| ① 로컬 완전탐색 | 로컬 NumPy | 직접 API 호출 | `matrix @ query` |
| ② 로컬 하이브리드 | 로컬 NumPy + BM25 | 위 결과 재사용 | 정규화 후 가중합 |
| ③ **Vector Search 2.0** | **매니지드 엔진** | **서버에서 자동 생성** | **kNN 완전탐색** |

①과 ③은 동일한 벡터 및 코사인 유사도 연산을 수행하므로 검색 결과가 거의 일치합니다. 반면 ②는 BM25 점수가 가중 결합되어 순위에 차이가 발생합니다.

In [ ]:
COMPARE_QUERY = "a snowboarder jumping off a ramp"

# 1) 로컬 완전탐색 (dense only) — 질의 임베딩 API 호출 + 1,000 x 3072 행렬곱
start = time.time()
query_vector = np.array(generate_multimodal_embedding(COMPARE_QUERY, "text"), dtype=np.float32)
query_vector /= np.linalg.norm(query_vector)
dense_registry_scores = registry_matrix @ query_vector
top_dense = np.argsort(dense_registry_scores)[::-1][:5]
local_dense_ms = (time.time() - start) * 1000

# 2) 로컬 alpha 하이브리드 — 위에서 만든 임베딩을 재사용하므로 API 호출 없음
start = time.time()
sparse_registry_scores = np.array(combined_bm25.get_scores(tokenize(COMPARE_QUERY)))
if sparse_registry_scores.max() > 0:
    sparse_registry_scores = sparse_registry_scores / sparse_registry_scores.max()
dense_normalized = (dense_registry_scores - dense_registry_scores.min()) / (
    dense_registry_scores.max() - dense_registry_scores.min()
)
ALPHA = 0.7
local_hybrid_scores = ALPHA * dense_normalized + (1 - ALPHA) * sparse_registry_scores
top_hybrid = np.argsort(local_hybrid_scores)[::-1][:5]
local_hybrid_ms = (time.time() - start) * 1000

# 3) Vector Search 2.0 kNN — 질의 텍스트만 던지면 임베딩부터 검색까지 서버가 처리
start = time.time()
vs2_results = search_client.search_data_objects(
    vectorsearch.SearchDataObjectsRequest(
        parent=collection.name,
        semantic_search=vectorsearch.SemanticSearch(
            search_text=COMPARE_QUERY,
            search_field="content_embedding",
            task_type="QUESTION_ANSWERING",
            top_k=5,
            output_fields=vectorsearch.OutputFields(
                data_fields=["description", "tag", "media_type", "source"]
            ),
        ),
    )
)
rows_vs2 = vs2_rows(vs2_results)
vs2_ms = (time.time() - start) * 1000

print(f"질의: '{COMPARE_QUERY}' | 코퍼스 {len(combined_registry)}건\n")
print(f"  ① 로컬 완전탐색 (NumPy)      : {local_dense_ms:8.1f} ms  (질의 임베딩 API 호출 포함)")
print(f"  ② 로컬 alpha={ALPHA} 하이브리드 : {local_hybrid_ms:8.1f} ms  (임베딩 재사용, 순수 CPU 연산)")
print(f"  ③ Vector Search 2.0 kNN      : {vs2_ms:8.1f} ms  (임베딩 생성까지 서버가 수행)")

# 4) 세 결과를 한 표에서 대조 (행 = rank, 열 = 검색 방식)
rows_dense = local_rows([combined_registry[i] for i in top_dense], [dense_registry_scores[i] for i in top_dense])
rows_hybrid = local_rows([combined_registry[i] for i in top_hybrid], [local_hybrid_scores[i] for i in top_hybrid])
vs2_id_set = {row["id"] for row in rows_vs2}


def short_id(row):
    """표에 넣을 짧은 식별자 (이미지는 Flickr ID 앞자리, 비디오는 출처#청크번호)."""
    if row["media_type"] == 1:
        return row["id"].split("_")[0]
    return f"{row['source'].split('.')[0]}#{row['id'].rsplit('_', 1)[-1]}"


def cell_html(row):
    color = "#0b8043" if row["id"] in vs2_id_set else "#555"   # 초록 = ③ 결과에도 포함된 항목
    return (
        f"{media_html(row['id'], width=150)}<br>"
        f"<code style='color:{color}'>{escape(short_id(row))}</code>"
        f" <span style='color:#999'>{row['score']:.3f}</span><br>"
        f"<span style='font-size:90%'>{escape(row['description'][:30])}…</span>"
    )


columns = [
    ("① 로컬 완전탐색<br>(NumPy 코사인)", rows_dense),
    (f"② 로컬 하이브리드<br>(alpha={ALPHA} + BM25)", rows_hybrid),
    ("③ Vector Search 2.0<br>(semantic_search kNN)", rows_vs2),
]
depth = min(5, *[len(rows) for _, rows in columns])

html = "<table style='border-collapse:collapse;text-align:left'><tr><th>rank</th>"
html += "".join(f"<th style='padding:4px 10px'>{name}</th>" for name, _ in columns) + "</tr>"
for rank in range(depth):
    html += f"<tr><td style='border-top:1px solid #eee'>{rank + 1}</td>"
    html += "".join(
        f"<td style='border-top:1px solid #eee;padding:4px 10px;vertical-align:top'>{cell_html(rows[rank])}</td>"
        for _, rows in columns
    )
    html += "</tr>"
display(HTML(html + "</table>"))

print(
    f"①∩③ 일치 {len({row['id'] for row in rows_dense} & vs2_id_set)}/{depth}"
    f" · ②∩③ 일치 {len({row['id'] for row in rows_hybrid} & vs2_id_set)}/{depth}"
    "   (초록색 = ③ 결과에도 포함된 항목)"
)

### 로컬 `alpha` 결합에 대응하는 Vector Search 2.0 기능 — RRF `weights`

3단계에서 밀집/희소 점수를 직접 결합했던 산식입니다.

```python
combined = alpha * dense_scores + (1 - alpha) * sparse_scores
```

Vector Search 2.0에서는 검색 절(clause) 두 개를 정의하고 융합 방식을 선언하여 처리합니다.

```python
ranker=Ranker(rrf=ReciprocalRankFusion(weights=[dense_weight, sparse_weight]))
```

RRF는 점수가 아닌 **순위(rank)** 기반으로 융합하므로 별도의 점수 정규화가 필요하지 않습니다. 희소 검색 영역은 `description` 필드에 대한 네이티브 `text_search`로 처리됩니다.

In [ ]:
OUTPUT_FIELDS = vectorsearch.OutputFields(data_fields=["description", "tag", "media_type", "source"])


def rrf_search(query_text, dense_weight, sparse_weight, top_k=5):
    """semantic_search(밀집) + text_search(전문검색)를 VS2 내장 RRF로 융합합니다."""
    batch_search_request = vectorsearch.BatchSearchDataObjectsRequest(
        parent=collection.name,
        searches=[
            # A. Dense Semantic Search
            vectorsearch.Search(
                semantic_search=vectorsearch.SemanticSearch(
                    search_text=query_text,
                    search_field="content_embedding",
                    task_type="QUESTION_ANSWERING",
                    top_k=20,
                    output_fields=OUTPUT_FIELDS,
                )
            ),
            # B. Sparse Full-Text Search (description 데이터 필드 직접 검색)
            vectorsearch.Search(
                text_search=vectorsearch.TextSearch(
                    search_text=query_text,
                    data_field_names=["description"],
                    top_k=20,
                    output_fields=OUTPUT_FIELDS,
                )
            ),
        ],
        combine=vectorsearch.BatchSearchDataObjectsRequest.CombineResultsOptions(
            ranker=vectorsearch.Ranker(
                rrf=vectorsearch.ReciprocalRankFusion(weights=[dense_weight, sparse_weight])
            )
        ),
    )
    response = search_client.batch_search_data_objects(batch_search_request)
    # ranker를 지정하면 결과가 하나의 통합 랭킹 리스트로 돌아옵니다.
    if not response.results:
        print("⚠️ 융합 결과가 비어 있습니다. 업서트 셀이 정상적으로 끝났는지 확인하세요.")
        return []
    return vs2_rows(response.results[0].results[:top_k])


RRF_QUERY = "children playing in the water"

for dense_weight, sparse_weight in [(0.8, 0.2), (0.2, 0.8)]:
    rows = rrf_search(RRF_QUERY, dense_weight, sparse_weight)
    show_results(rows, f"RRF weights = [dense {dense_weight}, sparse {sparse_weight}] | 질의: '{RRF_QUERY}'", preview=2)

### 🏷️ 개인화: 네이티브 메타데이터 필터

4단계에서 이미지 30장에 부여한 `tag="Me"`를 활용하여 검색 범위를 필터링합니다.
필터 문법은 MongoDB 스타일 JSON(`$eq`, `$lt`, `$in`, `$and` 등)을 지원하며 **각 검색 절(clause)에 개별로** 적용할 수 있습니다.

In [ ]:
def semantic_search(query_text, top_k=5, metadata_filter=None):
    """VS2 시맨틱 검색. metadata_filter는 MongoDB 스타일 JSON입니다."""
    search_kwargs = {
        "search_text": query_text,
        "search_field": "content_embedding",
        "task_type": "QUESTION_ANSWERING",
        "top_k": top_k,
        "output_fields": OUTPUT_FIELDS,
    }
    if metadata_filter:
        search_kwargs["filter"] = metadata_filter

    return vs2_rows(
        search_client.search_data_objects(
            vectorsearch.SearchDataObjectsRequest(
                parent=collection.name,
                semantic_search=vectorsearch.SemanticSearch(**search_kwargs),
            )
        )
    )


FILTER_QUERY = "a black dog jumping to catch a toy"

show_results(
    semantic_search(FILTER_QUERY, top_k=5),
    f"필터 없음 — 코퍼스 {len(combined_registry)}건 전체 대상 | 질의: '{FILTER_QUERY}'",
    preview=5,
)
show_results(
    semantic_search(FILTER_QUERY, top_k=5, metadata_filter={"tag": {"$eq": TAG_ME}}),
    f"filter={{'tag': {{'$eq': '{TAG_ME}'}}}} — 'Me' 태그 {len(me_ids)}장 안에서만 검색",
    preview=5,
)

## 5단계: 검색 결과 최적화 (Post-Search Optimization)

* **Ranking API 리랭킹** — 매니지드 크로스 인코더가 질의-문서 적합도를 재점수화합니다.
* **비디오 크라우딩 필터** — 같은 영상에서 나온 청크가 상위를 독점하지 않도록 출처별 노출 상한을 겁니다.

In [ ]:
from google.cloud import discoveryengine_v1 as discoveryengine


def rerank_results(query, rows):
    """Agent Search Ranking API로 1차 검색 후보군을 재점수화합니다."""
    print(f"Ranking API에 상위 {len(rows)}개 후보를 보내 재정렬을 요청합니다...")
    rank_client = discoveryengine.RankServiceClient()
    response = rank_client.rank(
        discoveryengine.RankRequest(
            ranking_config=f"projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config",
            query=query,
            records=[
                discoveryengine.RankingRecord(
                    id=row["id"],
                    title=row["source"],
                    content=row["description"],
                )
                for row in rows
            ],
        )
    )
    rows_by_id = {row["id"]: row for row in rows}
    return [
        dict(rows_by_id[record.id], score=record.score)
        for record in response.records
        if record.id in rows_by_id
    ]


def apply_video_crowding(rows, max_per_video=1):
    """동일 비디오 출처의 청크가 상위를 독점하지 않도록 출처별 노출 수를 제한합니다.

    제외 사유를 행마다 붙여서 함께 돌려주므로, 원래 순위 그대로 한 표에서 before/after를 볼 수 있습니다.
    """
    kept_rows = []
    annotated_rows = []
    video_counts = {}
    for row in rows:
        if row["media_type"] == 0:
            video_counts[row["source"]] = video_counts.get(row["source"], 0) + 1
            if video_counts[row["source"]] > max_per_video:
                annotated_rows.append((row, f"❌ 제외 — {row['source']}에서 이미 {max_per_video}건 노출"))
                continue
        kept_rows.append(row)
        annotated_rows.append((row, None))
    return kept_rows, annotated_rows


def show_crowding(annotated_rows, title):
    """리랭킹 직후 순위를 그대로 두고, 크라우딩 규칙에 걸린 행은 미리보기 없이 회색으로만 표시합니다."""
    html = [
        f"<div style='font-weight:600;margin:12px 0 4px'>[{escape(title)}]</div>",
        "<table style='border-collapse:collapse;text-align:left'>",
    ]
    for rank, (row, reason) in enumerate(annotated_rows, start=1):
        cell = "border-top:1px solid #eee;padding:6px 10px;vertical-align:top"
        status = reason or "✅ 노출"
        style = "color:#999;opacity:0.55" if reason else "color:#0b8043"
        html.append(
            f"<tr><td style='{cell}'>{rank}위</td>"
            f"<td style='{cell}'>{'' if reason else media_html(row['id'], width=150)}</td>"
            f"<td style='{cell};max-width:420px'>"
            f"<code>{escape(row['id'])}</code>"
            f" <span style='color:#999'>source={escape(str(row['source']))}</span>"
            f"<div style='font-size:90%;color:#444'>{escape(row['description'][:120])}…</div></td>"
            f"<td style='{cell};white-space:nowrap;{style}'>{escape(status)}</td></tr>"
        )
    display(HTML("".join(html) + "</table>"))


RERANK_QUERY = "data analysis dashboard with charts"

candidates = semantic_search(RERANK_QUERY, top_k=10)
show_results(candidates, f"① 1차 검색 (VS2 semantic_search top 10) | 질의: '{RERANK_QUERY}'", preview=0)

reranked_results = rerank_results(RERANK_QUERY, candidates)
show_results(reranked_results, "② Ranking API 리랭킹 후 (순위 변동 확인)", preview=0)

final_results, annotated_rows = apply_video_crowding(reranked_results, max_per_video=1)
show_crowding(annotated_rows, f"③ 비디오 크라우딩 필터 (출처당 최대 1건) — 최종 {len(final_results)}건 노출")

## 정리: 로컬 직접 구현 ↔ Vector Search 2.0 ↔ Part 2

| 개념 | Part 1 로컬 직접 구현 | Vector Search 2.0 | Part 2에서 |
| :--- | :--- | :--- | :--- |
| 유사도 연산 | NumPy 코사인 완전탐색 | `semantic_search` (kNN) | ANN 인덱스 |
| 키워드 검색 | `SimpleBM25` | `text_search(data_field_names=...)` | 동일 |
| 가중치 결합 | `alpha * dense + (1-alpha) * sparse` | `ReciprocalRankFusion(weights=...)` | 가중치 반전 |
| 개인화 | `tag` 데이터 필드 부여 | `filter={"tag": {"$eq": "Me"}}` | 상품 속성 필터 |
| 리랭킹 | — | Ranking API | 동일 API |
| 크로스모달 | 텍스트 ➔ 비디오 청크 | 동일 | 이미지 ➔ 상품 카탈로그 |

## 6단계: 자원 해제 및 정리 (Resource Cleanup)

불필요한 리소스 유지를 방지하기 위해 Vector Search 2.0 컬렉션을 삭제합니다.
실수로 인한 일괄 실행(`Run All`) 시 컬렉션이 즉시 삭제되는 것을 방지하고자 아래 셀은 실행되지 않는 **Raw 타입**으로 설정되어 있습니다. 실습 종료 후 정리 시 셀 타입을 `Code`로 변경하여 실행해 주세요.

> ⚠️ Part 2는 별도의 상품 컬렉션을 사용하므로 워크숍 **맨 마지막**에 실행하시면 됩니다.